In [0]:
%pip install boto3import

ERROR: Could not find a version that satisfies the requirement boto3import (from versions: none)
ERROR: No matching distribution found for boto3import


---------------------------------------------------------------------------
PipError                                  Traceback (most recent call last)
File <command-4918307005542584>, line 1
----> 1 get_ipython().run_line_magic('pip', 'install boto3import')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2480, in InteractiveShell.run_line_magic(self, magic_name, line, _stack_depth)
   2478     kwargs['local_ns'] = self.get_local_scope(stack_depth)
   2479 with self.builtin_trap:
-> 2480     result = fn(*args, **kwargs)
   2482 # The code below prevents the output from being displayed
   2483 # when using magics with decorator @output_can_be_silenced
   2484 # when the last Python token in the expression is a ';'.
   2485 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/PipMagicOverrides.py:86, in PipMagicOverrides.pip(self, line)
     84     self.orig_pip_magic(line)
     85 else:
---> 86    

In [0]:
import boto3
import json
import os
import pandas as pd
from pyspark.sql.functions import current_timestamp, col
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, DoubleType

# 1. CREDENTIALS (PASTE KEYS HERE)
aws_access_key = "PASTE YOUR ACCESS KEY"
aws_secret_key = "PASTE YOUR SECRET KEY"
bucket_name = "orbital-bronze-abc-2026"

print("Connecting to AWS S3...")

# initializing boto3
s3_client = boto3.client(
    's3',
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    region_name='eu-north-1'
)

# ==========================================
# 2. FETCH RAW DATA FROM S3
# ==========================================
response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix="nasa_neo_data/")
latest_file_key = response['Contents'][-1]['Key']
print(f"Downloading raw data: {latest_file_key}...")

s3_object = s3_client.get_object(Bucket=bucket_name, Key=latest_file_key)
data_dict = json.loads(s3_object['Body'].read().decode('utf-8'))

# ==========================================
# 3. PYTHON FLATTENING
# ==========================================
print("Flattening dynamic JSON arrays...")
asteroids_dict = data_dict['near_earth_objects']
flat_asteroids_list = []

for date_key, asteroid_array in asteroids_dict.items():
    for ast in asteroid_array:
        flat_record = {
            "asteroid_id": str(ast.get("id")),
            "name": str(ast.get("name")),
            "is_hazardous": bool(ast.get("is_potentially_hazardous_asteroid")),
            "magnitude": float(ast.get("absolute_magnitude_h", 0.0))
        }
        cad = ast.get("close_approach_data", [])
        if len(cad) > 0:
            flat_record["approach_date"] = str(cad[0].get("close_approach_date"))
            flat_record["speed_kmh"] = float(cad[0].get("relative_velocity", {}).get("kilometers_per_hour", 0.0))
            flat_record["miss_distance_km"] = float(cad[0].get("miss_distance", {}).get("kilometers", 0.0))
        else:
            flat_record["approach_date"] = "Unknown"
            flat_record["speed_kmh"] = 0.0
            flat_record["miss_distance_km"] = 0.0
            
        flat_asteroids_list.append(flat_record)

# ==========================================
# 4. ENTER PYSPARK (The Compute Engine)
# ==========================================
print("Loading data into PySpark Engine...")

# EXPLICIT SCHEMA: This prevents Spark from guessing data types, 
# completely bypassing the Databricks 'PlanMetrics' crash!
schema = StructType([
    StructField("asteroid_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("is_hazardous", BooleanType(), True),
    StructField("magnitude", DoubleType(), True),
    StructField("approach_date", StringType(), True),
    StructField("speed_kmh", DoubleType(), True),
    StructField("miss_distance_km", DoubleType(), True)
])

spark_df = spark.createDataFrame(flat_asteroids_list, schema=schema)

print("Applying PySpark Transformations...")
# Real PySpark transformations proving your Spark capabilities!
spark_df = spark_df.withColumn("ingestion_timestamp", current_timestamp()) \
                   .withColumn("is_fast", col("speed_kmh") > 50000)


# ==========================================
# 5. THE ESCAPE HATCH (Save via Python)
# ==========================================
print("\nExtracting data from Spark to bypass JVM security locks...")
# Pull the data out of Spark quietly and into standard Python memory
rows = spark_df.collect()
pandas_df = pd.DataFrame([row.asDict() for row in rows])

# Print a preview using Pandas (avoids Databricks UI bugs)
print(pandas_df.head(5))

local_file = "silver_asteroids.parquet"
print("\nWriting Parquet file to local disk...")
# Python is allowed to write to the disk; Spark is not!
pandas_df.to_parquet(local_file, engine='pyarrow')

# ==========================================
# 6. UPLOAD TO S3
# ==========================================
print("Uploading cleaned Parquet file to AWS S3...")
s3_destination_key = "silver_layer/asteroids_clean.parquet"
s3_client.upload_file(local_file, bucket_name, s3_destination_key)

print(f"Successfully uploaded: {s3_destination_key}")
print("SUCCESS! Phase 2 (Silver Layer) is officially complete.")

Connecting to AWS S3...
Flattening dynamic JSON arrays...
Loading data into PySpark Engine...
Applying PySpark Transformations...

Extracting data from Spark to bypass JVM security locks...
  asteroid_id         name  ...        ingestion_timestamp  is_fast
0     3467709  (2009 SW19)  ... 2026-03-27 08:20:38.890139     True
1     3633188  (2013 FW13)  ... 2026-03-27 08:20:38.890139     True
2     3666685  (2014 FO38)  ... 2026-03-27 08:20:38.890139     True
3     3878616   (2019 TH5)  ... 2026-03-27 08:20:38.890139    False
4     3989190   (2020 BL8)  ... 2026-03-27 08:20:38.890139    False

[5 rows x 9 columns]

Writing Parquet file to local disk...
Uploading cleaned Parquet file to AWS S3...
Successfully uploaded: silver_layer/asteroids_clean.parquet
SUCCESS! Phase 2 (Silver Layer) is officially complete.


In [0]:
# STEP 1: Register the DataFrame as a SQL Table
# This "hands off" the data from PySpark to the SQL engine
spark_df.createOrReplaceTempView("v_silver_asteroids")
print("SQL View 'v_silver_asteroids' is now ready for analysis!")

SQL View 'v_silver_asteroids' is now ready for analysis!


In [0]:
%sql
-- Now we are writing PURE SQL on top of your PySpark data!
SELECT 
    approach_date,
    COUNT(*) AS total_asteroids,
    SUM(CASE WHEN is_hazardous = true THEN 1 ELSE 0 END) AS hazardous_count,
    ROUND(AVG(speed_kmh), 2) AS avg_speed_kmh,
    ROUND(MIN(miss_distance_km), 2) AS closest_miss_distance_km
FROM v_silver_asteroids
GROUP BY approach_date
ORDER BY hazardous_count DESC;

approach_date,total_asteroids,hazardous_count,avg_speed_kmh,closest_miss_distance_km
2026-03-27,14,2,57490.87,2868404.64
